# 🏏 IPL — Batter vs Spin Bowling · Thesis Visualisations

Standalone notebook that produces the RQ1/RQ2 figures for the thesis findings chapter.
Reads directly from `../csv/`, exactly like `IPL_Spin_Prediction_Model.ipynb`.

Figures produced:
1. Venue risk–reward scatter (RQ2 — venue effect on batting vs spin)
2. Venue ranking bar chart (top/bottom 8 by wicket rate)
3. Strike rate & dismissal rate by spin bowling style
4. Predicted vs actual runs — calibration + error distribution (RQ1)


## 📦 Step 1 — Imports & Config

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.edgecolor': '#333333',
    'axes.labelcolor': '#222222',
    'text.color': '#222222',
    'axes.titleweight': 'bold',
})

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_FOLDER = '../csv'              # ← points to csv/ folder (notebook is in notebooks/)
FIG_FOLDER  = '../figures'          # ← where PNGs get saved
os.makedirs(FIG_FOLDER, exist_ok=True)


## 🏟️ Step 2 — Figure 1: Venue Risk–Reward Scatter (RQ2)

Loads `venue_features.csv` (built in `IPL_Spin_Prediction_Model.ipynb`, Step 5b) and plots each
venue's spin economy against its spin wicket rate, split into quadrants around the median.
This is the primary figure for RQ2 — it lets the thesis name specific bowler-friendly and
batter-friendly venues rather than asserting the venue effect abstractly.

In [ ]:
vf = pd.read_csv(os.path.join(DATA_FOLDER, 'venue_features.csv'))

med_wkt = vf['venue_spin_wkt_rate'].median()
med_eco = vf['venue_spin_economy'].median()

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(vf['venue_spin_economy'], vf['venue_spin_wkt_rate'] * 100,
           s=90, color='#2b6cb0', edgecolor='white', linewidth=0.8, zorder=3, alpha=0.85)

ax.axvline(med_eco, color='#999999', linestyle='--', linewidth=1, zorder=1)
ax.axhline(med_wkt * 100, color='#999999', linestyle='--', linewidth=1, zorder=1)

# Label the most extreme / interesting venues only, to avoid clutter
label_set = set()
label_set.update(vf.nlargest(5, 'venue_spin_wkt_rate')['venue'])
label_set.update(vf.nsmallest(4, 'venue_spin_economy')['venue'])
label_set.update(vf.nlargest(3, 'venue_spin_economy')['venue'])
label_set.update(vf.nsmallest(3, 'venue_spin_wkt_rate')['venue'])

for _, row in vf.iterrows():
    if row['venue'] in label_set:
        ax.annotate(row['venue'], (row['venue_spin_economy'], row['venue_spin_wkt_rate'] * 100),
                    fontsize=7.6, xytext=(5, 4), textcoords='offset points', color='#333333')

xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()
ax.text(xmax - 0.01, ymax - 0.15, 'High risk, low scoring\n(bowler-friendly)',
        ha='right', va='top', fontsize=8.5, color='#7a2e2e', style='italic')
ax.text(xmin + 0.01, ymax - 0.15, 'High risk, high scoring\n(volatile)',
        ha='left', va='top', fontsize=8.5, color='#7a5a2e', style='italic')
ax.text(xmax - 0.01, ymin + 0.15, 'Low risk, high scoring\n(batter-friendly)',
        ha='right', va='bottom', fontsize=8.5, color='#2e6b3e', style='italic')
ax.text(xmin + 0.01, ymin + 0.15, 'Low risk, low scoring\n(defensive)',
        ha='left', va='bottom', fontsize=8.5, color='#333333', style='italic')

ax.set_xlabel('Venue spin economy (runs conceded per ball)')
ax.set_ylabel('Venue spin wicket rate (%)')
ax.set_title(f'Venue-Level Risk–Reward Profile Against Spin Bowling\n(n = {len(vf)} venues, minimum 50 spin deliveries each)',
             fontsize=12)
ax.grid(alpha=0.25, zorder=0)
fig.tight_layout()
fig.savefig(os.path.join(FIG_FOLDER, 'fig_venue_riskreward.png'), dpi=300, bbox_inches='tight')
plt.show()


## 📊 Step 3 — Figure 2: Venue Ranking (Top/Bottom 8 by Wicket Rate)

In [ ]:
vf_sorted = vf.sort_values('venue_spin_wkt_rate', ascending=True)
top8    = vf_sorted.tail(8)
bottom8 = vf_sorted.head(8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

ax1.barh(bottom8['venue'], bottom8['venue_spin_wkt_rate'] * 100, color='#38a169')
ax1.set_title('Lowest Spin Wicket Rate\n(most batter-friendly)', fontsize=11)
ax1.set_xlabel('Venue spin wicket rate (%)')

ax2.barh(top8['venue'], top8['venue_spin_wkt_rate'] * 100, color='#c53030')
ax2.set_title('Highest Spin Wicket Rate\n(most bowler-friendly)', fontsize=11)
ax2.set_xlabel('Venue spin wicket rate (%)')

fig.suptitle('IPL Venues Ranked by Spin Wicket-Taking Rate', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(os.path.join(FIG_FOLDER, 'fig_venue_ranking.png'), dpi=300, bbox_inches='tight')
plt.show()


## 🌀 Step 4 — Figure 3: Strike Rate & Dismissal Rate by Spin Bowling Style

Loads `batter_vs_spin_stats.csv` and aggregates across *all* batters, weighted by balls faced,
to compare how batters perform against each of the five spin bowling styles.

In [ ]:
bs = pd.read_csv(os.path.join(DATA_FOLDER, 'batter_vs_spin_stats.csv'))

agg = bs.groupby('bowler_spin_type').agg(
    total_balls=('balls', 'sum'),
    total_runs=('runs', 'sum'),
    total_dismissals=('dismissals', 'sum'),
).reset_index()
agg['sr']              = agg['total_runs'] / agg['total_balls'] * 100
agg['dismissal_rate']  = agg['total_dismissals'] / agg['total_balls'] * 100
agg = agg.sort_values('sr')

label_map = {
    'right-arm offbreak':     'Right-arm offbreak',
    'slow left-arm orthodox': 'Slow left-arm orthodox',
    'legbreak googly':        'Legbreak googly',
    'legbreak':                'Legbreak',
    'left-arm wrist-spin':    'Left-arm wrist-spin',
}
agg['label'] = agg['bowler_spin_type'].map(label_map)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(agg)))

ax1.barh(agg['label'], agg['sr'], color=colors)
ax1.set_xlabel('Aggregate strike rate')
ax1.set_title('Strike Rate by Spin Bowling Type', fontsize=11)
for i, (v, b) in enumerate(zip(agg['sr'], agg['total_balls'])):
    ax1.text(v + 0.5, i, f'{v:.1f}  (n={b:,})', va='center', fontsize=8)

ax2.barh(agg['label'], agg['dismissal_rate'], color=colors)
ax2.set_xlabel('Dismissal rate (%)')
ax2.set_title('Dismissal Rate by Spin Bowling Type', fontsize=11)
for i, v in enumerate(agg['dismissal_rate']):
    ax2.text(v + 0.05, i, f'{v:.2f}%', va='center', fontsize=8)

fig.suptitle('Batter Performance Against Different Spin Bowling Styles (All IPL Batters)',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(os.path.join(FIG_FOLDER, 'fig_spintype_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()


## ✅ Step 5 — Figure 4: Model Calibration (RQ1)

Loads `validation_batter_match.csv` and plots predicted vs. actual runs at the batter-match
level, alongside the prediction-error distribution. Reports overall match-level accuracy rate
and mean absolute error — note these are coarser than the ball-level MAE reported elsewhere,
since they aggregate to one prediction per batter per match rather than per ball.

In [ ]:
vm = pd.read_csv(os.path.join(DATA_FOLDER, 'validation_batter_match.csv'))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax1 = axes[0]
ax1.scatter(vm['actual_runs'], vm['predicted_runs'], s=10, alpha=0.35, color='#2b6cb0')
lims = [0, max(vm['actual_runs'].max(), vm['predicted_runs'].max())]
ax1.plot(lims, lims, color='#c53030', linestyle='--', linewidth=1.3, label='Perfect prediction')
ax1.set_xlabel('Actual runs')
ax1.set_ylabel('Predicted runs')
ax1.set_title(f'Predicted vs Actual Runs\n(per batter-match, n = {len(vm)})', fontsize=11)
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(alpha=0.25)

ax2 = axes[1]
ax2.hist(vm['run_error'], bins=40, color='#4a5568', edgecolor='white')
mean_err = vm['run_error'].mean()
ax2.axvline(0, color='#c53030', linestyle='--', linewidth=1.3, label='Zero error')
ax2.axvline(mean_err, color='#d69e2e', linestyle='--', linewidth=1.3, label=f'Mean error: {mean_err:.2f}')
ax2.set_xlabel('Prediction error (predicted − actual runs)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of Prediction Error', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.25)

fig.suptitle('Model Calibration on Held-Out Batter-Match Validation Set', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(os.path.join(FIG_FOLDER, 'fig_calibration.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"Match-level accuracy rate (correct_prediction=True): {vm['correct_prediction'].mean():.4f}")
print(f"Match-level mean absolute error: {vm['abs_error'].mean():.4f} runs")


## ⚖️ Step 6 — Figure 5: Prediction Volatility by Sample Size (RQ2)

Tests whether prediction reliability depends on how much spin-specific data a batter has.
Rather than raw absolute error (which scales with a batter's typical run output and is
therefore not a fair reliability measure on its own), this uses **relative error**
(`abs_error / mean_actual_runs`) grouped into sample-size bins.

This is the evidence for RQ2 — it shows that low-sample predictions are far more *volatile*
(wide spread of relative error), which justifies presenting them with an explicit confidence
rating rather than as a bare number with the same apparent certainty as a high-sample
prediction.

In [ ]:
vm = pd.read_csv(os.path.join(DATA_FOLDER, 'validation_batter_match.csv'))

agg = vm.groupby('batter').agg(
    total_balls=('balls_faced', 'sum'),
    mean_actual_runs=('actual_runs', 'mean'),
    mean_abs_error=('abs_error', 'mean'),
).reset_index()

# Relative error avoids the scale bias of raw absolute error
# (prolific batters naturally have larger absolute errors simply because they score more runs)
agg['mean_actual_runs_safe'] = agg['mean_actual_runs'].replace(0, 0.5)
agg['relative_error_pct'] = agg['mean_abs_error'] / agg['mean_actual_runs_safe'] * 100
agg['relative_error_pct_clipped'] = agg['relative_error_pct'].clip(upper=150)  # clip for plot readability only

bin_edges  = [0, 15, 30, 60, 120, 10000]
bin_labels = ['≤15', '16–30', '31–60', '61–120', '>120']
agg['bin'] = pd.cut(agg['total_balls'], bins=bin_edges, labels=bin_labels)

counts = agg['bin'].value_counts().reindex(bin_labels)
labels_with_n = [f'{b}\n(n={counts[b]} batters)' for b in bin_labels]
data_by_bin = [agg[agg['bin'] == b]['relative_error_pct_clipped'].dropna().values for b in bin_labels]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(
    data_by_bin, tick_labels=labels_with_n, patch_artist=True, showfliers=True,
    flierprops=dict(marker='o', markersize=3, alpha=0.4, markerfacecolor='#c53030', markeredgecolor='none')
)
colors = ['#c53030', '#dd6b20', '#d69e2e', '#38a169', '#2b6cb0']
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)

ax.set_xlabel('Total balls faced against spin in validation set (sample size)')
ax.set_ylabel('Relative prediction error (%) per batter\n(clipped at 150% for readability)')
ax.set_title('Prediction Volatility Shrinks as Sample Size Increases\n(mean relative error per batter, grouped by balls faced)',
             fontsize=12)
ax.grid(alpha=0.25, axis='y')
fig.tight_layout()
fig.savefig(os.path.join(FIG_FOLDER, 'fig_confidence_justification.png'), dpi=300, bbox_inches='tight')
plt.show()

# Summary stats for the write-up
summary = agg.groupby('bin', observed=True)['relative_error_pct'].agg(
    n='count', std='std', median='median'
).reindex(bin_labels)
print(summary.round(1))


## 📝 Notes

- All four PNGs are saved to `../figures/` at 300 DPI, ready to insert into the thesis document.
- Figure 1 needs `venue_features.csv` — regenerate it in `IPL_Spin_Prediction_Model.ipynb`
  (Step 5b) if it's missing or stale.
- Figure 4's match-level MAE will differ from the ball-level MAE reported in the training
  notebook — state both explicitly in the thesis, since they measure different granularities.
- Figure 5's relative-error clipping (150%) is for plot readability only — report the
  unclipped std values printed beneath the figure in the thesis text, not the clipped ones.
- If you later add a venue column to `validation_batter_match.csv` (join on `match_id` via
  `Match_Info.csv`), a fifth figure — prediction accuracy broken down by venue — becomes
  possible and would strengthen the RQ2 discussion further.
